<a href="https://colab.research.google.com/github/kaxmarc/data-management/blob/main/individual-excercise/week-2%20/exercise-2/05_individual_exercise_data_quality_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Individual Exercise: Mini Data Quality Audit

Homework notebook for Week 2.


## Instructions

Perform a mini data quality audit and submit this completed notebook.


In [21]:
from pathlib import Path
import pandas as pd
import numpy as np
df = pd.read_csv('https://raw.githubusercontent.com/kaxmarc/data-management/refs/heads/main/individual-excercise/week-2/exercise-2/week2_customer_transactions_messy.csv')
print('Shape:', df.shape)
df.head()


Shape: (11, 9)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09


## Required tasks

1. describe the dataset briefly
2. identify issues by dimension
3. compute at least 3 KPIs
4. define at least 3 validation rules
5. create a short audit summary
6. recommend cleaning steps for next chapter (do not implement full pipelines)


## Task 1 - Dataset description

The dataset contains **10 unique financial transactions**, or 11 records in total, including 1 duplicate. It covers European and North American markets.  

Each record represents a single customer payment.

**Potential business applications:** monitoring payment processing, sales reporting, fraud detection, analysis of customer behaviour (e.g. payment history) and regulatory compliance.


| Column           | Type   | Description                                     |
| :--------------- | :----- | :---------------------------------------------- |
| transaction_id   | String | Unique identifier for each transaction          |
| customer_id      | String | Reference to the purchasing customer            |
| transaction_date | Date   | Date the transaction was executed               |
| amount           | Float  | Monetary value in the given currency            |
| currency         | String | currency code (e.g. EUR, USD)          |
| payment_method   | String | Channel used (card, bank_transfer, cash)        |
| status           | String | Lifecycle state (completed, pending, cancelled) |
| region           | String | country code                 |
| last_updated     | Date   | Timestamp of last record modification           |

In [22]:
df.info()

df.describe(include='all').transpose()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    11 non-null     object 
 1   customer_id       10 non-null     object 
 2   transaction_date  11 non-null     object 
 3   amount            10 non-null     float64
 4   currency          11 non-null     object 
 5   payment_method    10 non-null     object 
 6   status            11 non-null     object 
 7   region            11 non-null     object 
 8   last_updated      10 non-null     object 
dtypes: float64(1), object(8)
memory usage: 924.0+ bytes


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
transaction_id,11,10,T0006,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,10,9,C105,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
transaction_date,11,8,2026-01-07,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
amount,10.0,NaN,NaN,NaN,100056.457,316207.939002,-35.0,19.99,49.55,112.8725,1000000.0
currency,11,3,EUR,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
payment_method,10,4,card,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
status,11,3,completed,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,11,5,DE,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
last_updated,10,9,2026-04-15,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Task 2 - Issues by dimension


In [23]:
issue_table = pd.DataFrame([
    ('Missing customer id', 'Completeness', 'Impacts customer analytics'),
    ('Missing amount', 'Completeness', 'Impacts Value reports'),
    ('Missing payment method', 'Completeness', 'Impacts fraud investigation and methodological analysis'),
    ('Missing last updated', 'Completeness', 'Impacts Audit compliance'),
    ('Duplicate transaction id','Uniqueness','May double count revenue'),
    ('Invalid date','Validity','Deviations from the standard format cause parsing errors'),
    ('Invalid currency','Validity','Non-standard code breaks currency conversion and reporting processes'),
    ('Invalid payment method','Validity','Different spellings limit and hinder search options'),
    ('Invalid region code','Validity','Different spellings limit and hinder search options'),
    ('Invalid transfer value','Validity','Transaction outliers require plausibility checks'),
    ('Integrity deviation last updated','Integrity','Date invalid'),

], columns=['Issue','Dimension','Impact'])
issue_table


,Issue,Dimension,Impact
0,Missing customer id,Completeness,Impacts customer analytics
1,Missing amount,Completeness,Impacts Value reports
2,Missing payment method,Completeness,Impacts fraud investigation and methodological...
3,Missing last updated,Completeness,Impacts Audit compliance
4,Duplicate transaction id,Uniqueness,May double count revenue
5,Invalid date,Validity,Deviations from the standard format cause pars...
6,Invalid currency,Validity,Non-standard code breaks currency conversion a...
7,Invalid payment method,Validity,Different spellings limit and hinder search op...
8,Invalid region code,Validity,Different spellings limit and hinder search op...
9,Invalid transfer value,Validity,Transaction outliers require plausibility checks


## Task 3 - KPI calculations


In [24]:

# KPI 1: Completenes Rate
total_cells = df.shape[0]*df.shape[1]
missing_cells = df.isna().sum().sum()
completeness = 1-(missing_cells/total_cells)

# KPI 2: Duplicate Rate
dup_full = df.duplicated().mean()
dup_key = df.duplicated(subset=['transaction_id']).mean()

# KPI 3: Error Rate
n=len(df)
trx = pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed')
upd = pd.to_datetime(df['last_updated'], errors='coerce', format='mixed')
amount = pd.to_numeric(df['amount'], errors='coerce')
flags = pd.DataFrame({
'missing_customer': df['customer_id'].isna() | (df['customer_id'].astype(str).str.strip()==''),
'missing_amount': amount.isna(),
'missing_payment': df['payment_method'].isna() | (df['payment_method'].astype(str).str.strip()==''),
'missing_last_updated': df['last_updated'].isna(),
'invalid_amount': (amount<0) | (amount>10000),
'invalid_date': trx.isna() | upd.isna(),
'invalid_currency': ~df['currency'].isin(['EUR','USD']),
'invalid_payment': ~df['payment_method'].isin(['card','bank_transfer','cash']),
'invalid_region': ~df['region'].str.isupper(),
})
flags['record_error'] = flags.any(axis=1)
error_rate = flags['record_error'].mean()

# KPI 4: Consitency (Transaction Date Format)
cons_date = df['transaction_date'].str.match(r'^\d{4}-\d{2}-\d{2}$', na=False).sum()
cons_date_rate = cons_date / len(df)

# KPI Summary
overall = 0.30*completeness + 0.30*(1-error_rate) + 0.20*(1-dup_key) + 0.20*cons_date_rate

kpi = pd.DataFrame([
('Completeness Rate', completeness, 'Higher is better'),
('Duplication Rate (transaction_id)', dup_key, 'Lower is better'),
('Error Rate', error_rate, 'Lower is better'),
('Consitency Rate', cons_date_rate, 'Higher is better'),
('Overall Score', overall, 'Higher is better')
], columns=['KPI','Value','Direction'])
kpi['Value (%)']=(kpi['Value']*100).round(2)
kpi[['KPI','Value (%)','Direction']]

,KPI,Value (%),Direction
0,Completeness Rate,95.96,Higher is better
1,Duplication Rate (transaction_id),9.09,Lower is better
2,Error Rate,72.73,Lower is better
3,Consitency Rate,81.82,Higher is better
4,Overall Score,71.52,Higher is better


### Your KPI interpretation

**The error rate (72.73%)** is the most striking finding in this context. It points to a systemic quality issue rather than merely isolated incorrect entries.


## Task 4 - Validation rules


In [25]:
RELEVANT_FIELDS = ['transaction_id', 'customer_id', 'transaction_date', 'amount', 'currency', 'payment_method', 'status', 'region']

rules = {
    # Rule 1 Completeness: no important field may be NULL
    'relevant_fields_present':
        df[RELEVANT_FIELDS].isnull().any(axis=1),

    # Rule 2 Uniqueness: transaction_id must be unique
    'transaction_id_unique':
        df.duplicated(subset='transaction_id', keep=False),

    # Rule 3 Validity: amount must be positive
    'amount_positive': (
        df['amount'].notna()
        & (df['amount'] <= 0)
        & df['status'].isin({'completed', 'pending'})
    ),

    # Rule 4 Validity: transaction_date must "YYYY-MM-DD ISO format"
    'transaction_date_iso_format': (
        ~df['transaction_date'].str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
        & df['transaction_date'].notna()
    ),

    # Rule 5 Validity: currency must be a recognised code
    'currency_valid': (
        ~df['currency'].isin(['EUR','USD'])
        & df['currency'].notna()
    ),
}

pd.DataFrame(
    {'affected_rows': {k: int(v.sum()) for k, v in rules.items()}},
    index=list(rules.keys())
)

,affected_rows
relevant_fields_present,3
transaction_id_unique,2
amount_positive,2
transaction_date_iso_format,2
currency_valid,1


## Task 5 - Audit summary


In [26]:
audit_summary = pd.DataFrame([
('Missing important field: customer_id', 'T0004', 'Critical', 'Escalate and resolve missing IDs before joins'),
('Missing important field: amount', 'T0009', 'Critical', 'Retrieve from source system logs; exclude from revenue aggregations until confirmed.'),
('Duplicate row (transaction T0006)', 'rows 7-8', 'Critical', 'Remove duplicates: Keep the first occurrence.'),
('Non-standard date format YYYY/MM/DD', 'T0002', 'High', 'Adjust to the standard format. Check the source of the error.'),
('Non-standard date format DD-MM-YYYY', 'T0003', 'High', 'Adjust to the standard format. Check the source of the error.'),
('Invalid currency code EURO (instead EUR)', 'T0005', 'High', 'Mapping via normalisation (EURO to EUR) and updating of the upstream input validation.'),
('Negative amount for completed transaction', 'T0003', 'High', 'Check with the finance department: if ‘Reimbursed’ is permitted, adjust the restriction.'),
('Zero amount', 'T0002', 'High', 'Check the cause. Possibly a test transaction or faulty pre-processing steps.'),
('Missing payment_method', 'T0010', 'High', 'Track and correct the transaction. Identify the cause.'),
('payment_method wrong case CARD', 'T0002', 'Medium', 'Set up the intercept function. Apply str.lower() normalisation at load time.'),
('region wrong case de', 'T0002', 'Medium', 'Set up the intercept function. Apply str.lower() normalisation at load time.'),
('Outlier amount 1000000 EUR', 'T0008', 'Medium', 'Check for plausibility. This may indicate an attempt at fraud or an incorrect entry.'),
('Missing last_updated', 'T0009', 'Low', 'Add the "Transaction_date" and check the cause.'),
], columns=['issue_type', 'affected_rows', 'severity', 'recommended_next_action'])

sev_order = {'Critical': 0, 'High': 1, 'Medium': 2, 'Low': 3}
audit_summary = (audit_summary
                 .assign(_s=audit_summary['severity'].map(sev_order))
                 .sort_values('_s').drop(columns='_s').reset_index(drop=True))

print(f'Total issues: {len(audit_summary)}')
print(audit_summary['severity'].value_counts().to_string())
print()
audit_summary

Total issues: 13
severity
High        6
Critical    3
Medium      3
Low         1



,issue_type,affected_rows,severity,recommended_next_action
0,Missing important field: customer_id,T0004,Critical,Escalate and resolve missing IDs before joins
1,Missing important field: amount,T0009,Critical,Retrieve from source system logs; exclude from...
2,Duplicate row (transaction T0006),rows 7-8,Critical,Remove duplicates: Keep the first occurrence.
3,Non-standard date format YYYY/MM/DD,T0002,High,Adjust to the standard format. Check the sourc...
4,Non-standard date format DD-MM-YYYY,T0003,High,Adjust to the standard format. Check the sourc...
5,Invalid currency code EURO (instead EUR),T0005,High,Mapping via normalisation (EURO to EUR) and up...
6,Negative amount for completed transaction,T0003,High,Check with the finance department: if ‘Reimbur...
7,Zero amount,T0002,High,Check the cause. Possibly a test transaction o...
8,Missing payment_method,T0010,High,Track and correct the transaction. Identify th...
9,payment_method wrong case CARD,T0002,Medium,Set up the intercept function. Apply str.lower...


## Task 6 - Recommended cleaning steps for next chapter

**Step 1: De-duplicate:** Drop exact duplicate rows, keeping the first occurrence

**Step 2: Standardise date formats:** Parse and convert all invalid dates to ISO Standard.

**Step 3_ Normalise text casing:** Apply '.str.upper()' to 'region' and 'currency'; '.str.lower()' to 'payment_method'.

**Step 4: Map non-standard currency codes:** Build a lookup dict 'EURO': 'EUR' and replace invalid codes.

**Step 5: Handle missing important fields:** Route records with NULL 'customer_id' or NULL 'amount' to a quarantine table for manual review.

**Step 6: Handle missing "optional" fields:** Impute payment_method = 'unknown'; refill 'last_updated' with 'transaction_date'.

**Step 7: Flag outliers (e.g. high values):** Add boolean 'is_amount_outlier' column.

**Step 8: Flag non-positive amounts:** Add 'amount_flag' column ('negative', 'zero', 'ok').


## Reflection questions

1. **Which KPI provided the clearest indication?**  
The **error rate (72.73%)** provided the clearest indication. This rate suggests a systemic problem at source rather than isolated incorrect entries. This finding alone justifies a comprehensive review of data validation at an early stage. At this point, the audit must be extended to the input systems.

2. **Which problem should be escalated first?**  
The three **critical** issues (missing customer_id, missing amount, duplicate) should be escalated first, as they either completely block financial processing (missing amount), violate fundamental data integrity (duplicate) or have regulatory consequences (missing customer_id).

3. **What additional metadata would improve this check?**  
Three additions would significantly improve the check:
   - **Data origin of the source system** specifies which upstream system generated the respective data record, enabling targeted corrections at the source.
   - **Timestamp for the data record** enables better root cause analysis within the pipeline.
   - **Schema version** explains why some rows use non-standardised formats (e.g. date)

## Bonus (recommended): write your own audit function

Create at least one helper function with clear parameters and docstring.
This demonstrates that your logic is reusable and understandable.


In [27]:
def summarize_rule_violations(rule_dictionary):
    """Summarize affected-row counts for each validation rule.

    Parameters
    ----------
    rule_dictionary : dict[str, pd.Series]
        Dictionary where keys are rule names and values are boolean masks.

    Returns
    -------
    pd.DataFrame
        Table with one row per rule and violation counts.
    """
    return pd.DataFrame({
        'rule_name': list(rule_dictionary.keys()),
        'affected_rows': [int(mask.sum()) for mask in rule_dictionary.values()],
        'violation_rate': [round(mask.mean() * 100, 1) for mask in rule_dictionary.values()],
    }).sort_values('affected_rows', ascending=False)

# Example usage with your `rules` dictionary:
summarize_rule_violations(rules)


,rule_name,affected_rows,violation_rate
0,relevant_fields_present,3,27.3
1,transaction_id_unique,2,18.2
2,amount_positive,2,18.2
3,transaction_date_iso_format,2,18.2
4,currency_valid,1,9.1


### Explain your function parameters

The parameter contains the created `rule_dictionary` (task 4), which has now been called here using a function. Changes to its contents have an immediate effect on the output. In this example, the `dict-Summary` is sorted by `affected_rows` and output with additional weighting.


## Submission checklist

- [X] Dataset described
- [X] Issues mapped to dimensions
- [X] At least 3 KPIs calculated
- [X] At least 3 validation rules defined
- [X] Audit summary completed
- [X] Cleaning recommendations proposed (not implemented)
